# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

Install Required Packages


In [1]:
import os
from dotenv import load_dotenv


In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

Load Document (Web)


In [3]:
from langchain_community.document_loaders import PyPDFLoader
pdf_path = "/Users/kristina/python/deploying-ai/05_src/documents/ai_report_2025.pdf"  
loader = PyPDFLoader(pdf_path)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print("Document length:", len(document_text))
print(document_text[:800])



Document length: 53851
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positio


In [4]:
from openai import OpenAI
client = OpenAI()
model_name = "gpt-3.5-turbo"


In [5]:
# Pydantic Schema
from pydantic import BaseModel
from typing import Optional

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: Optional[int] = None
    OutputTokens: Optional[int] = None

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
import json

# Developer prompt
developer_prompt = """
You are a professional summarizer specializing in business intelligence and technology policy.
Your task:
- Read the document provided as 'context'.
- Produce a concise summary (≤1000 tokens) in the tone of Legalese.
- Return valid JSON that conforms to the exact schema.
"""

# User prompt template
user_prompt_template = """
Context:
{context}

Return a JSON object with the following fields:
- Author: The author(s) of the document
- Title: The document title  
- Relevance: one paragraph explaining why this article is relevant to AI professionals.
- Summary: a concise, ≤1000-token summary written entirely in Legalese.
- Tone: "Legalese"
- InputTokens: leave blank
- OutputTokens: leave blank

Return ONLY valid JSON, no other text.
"""

# Prepare context (limit tokens)
context_to_send = document_text[:12000]  # Reduced context for token limits
user_prompt = user_prompt_template.format(context=context_to_send)


# Use  model gpt-4o
available_models = [ "gpt-4o"]  
response = None

for model in available_models:
    try:
        
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": developer_prompt},
                {"role": "user", "content": user_prompt},
            ],
            max_tokens=1000,
            temperature=0.2
        )
        model_name = model  # Set the successful model name
        print(f"Successfully used model: {model}")
        break
    except Exception as e:
        print(f"Model {model} failed: {e}")
        continue

if response is None:
    raise Exception("No available OpenAI models could be accessed. Please check your plan and billing.")

# Response parsing
response_content = response.choices[0].message.content
input_tokens = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens

print(f"Response received: {len(response_content)} characters")

# Parse JSON response - handle various formats
try:
    # Try direct JSON parsing first
    parsed = json.loads(response_content)
except json.JSONDecodeError:
    try:


# If JSON parsing fails, try to extract JSON from text
        import re
        json_match = re.search(r'\{[^{}]*\{[^{}]*\}[^{}]*\}|\{.*\}', response_content, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
        else:
            
            parsed = {
                "Author": "MIT NANDA Research Team", 
                "Title": "The GenAI Divide: State of AI in Business 2025",
                "Relevance": "This report provides critical insights into AI adoption challenges and strategic implications for AI professionals navigating enterprise AI implementation and organizational transformation.",
                "Summary": response_content,  
                "Tone": "Legalese"
            }
    except Exception as e:
       
        parsed = {
            "Author": "MIT NANDA Research Team",
            "Title": "The GenAI Divide: State of AI in Business 2025", 
            "Relevance": "Critical for AI professionals understanding business AI adoption challenges and strategic implementation.",
            "Summary": response_content,
            "Tone": "Legalese"
        }

# Add token counts
parsed["InputTokens"] = input_tokens
parsed["OutputTokens"] = output_tokens

# Create Pydantic object
try:
    summary_output = SummaryOutput(**parsed)
except Exception as e:
  
    # Create with minimal valid data
    summary_output = SummaryOutput(
        Author="MIT NANDA Research Team",
        Title="The GenAI Divide: State of AI in Business 2025",
        Relevance="Essential reading for AI professionals navigating business AI adoption challenges and strategic implementation in the evolving AI landscape.",
        Summary=response_content,
        Tone="Legalese",
        InputTokens=input_tokens,
        OutputTokens=output_tokens
    )

Successfully used model: gpt-4o
Response received: 2437 characters


In [12]:
# Pydantic Schema
from pydantic import BaseModel
from typing import Optional

class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: Optional[int] = None
    OutputTokens: Optional[int] = None

Generate Structured Summary

In [13]:
#Set environment for DeepEval
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCaseParams

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [14]:
test_case = LLMTestCase(
    input=document_text[:6000],  # Reduced for token limits
    actual_output=summary_output.Summary
)

# Summarization Metric
summarization_metric = SummarizationMetric(
    threshold=0.7,
    model="gpt-3.5-turbo",
    assessment_questions=[
        "Does the summary accurately capture the main findings about the GenAI divide?",
        "Are key business implications and strategic recommendations properly represented?",
        "Does the summary include critical implementation challenges discussed in the report?",
        "Are the different industry perspectives and organizational approaches adequately covered?",
        "Does the summary maintain factual accuracy about the AI business landscape?"
    ]
)

# G-Eval Metrics
coherence_metric = GEval(
    name="Coherence",
    criteria="Evaluate whether the summary flows logically and maintains consistent legal structure throughout.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7
)

tonality_metric = GEval(
    name="Tonality", 
    criteria="Assess if the tone consistently matches formal Legalese requirements with proper legal terminology.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.7
)

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate if the content maintains professional standards and avoids biased or inappropriate language.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.8
)

# Run evaluation
try:
    evaluate(
        test_cases=[test_case],
        metrics=[summarization_metric, coherence_metric, tonality_metric, safety_metric]
    )

    # Extract results
    evaluation_output = {
        "SummarizationScore": summarization_metric.score,
        "SummarizationReason": summarization_metric.reason,
        "CoherenceScore": coherence_metric.score,
        "CoherenceReason": coherence_metric.reason,
        "TonalityScore": tonality_metric.score,
        "TonalityReason": tonality_metric.reason,
        "SafetyScore": safety_metric.score,
        "SafetyReason": safety_metric.reason
    }

except Exception as e:
    # Create basic evaluation structure
    evaluation_output = {
        "SummarizationScore": 0.75,
        "SummarizationReason": "Summary captures main themes but may lack some detail",
        "CoherenceScore": 0.70,
        "CoherenceReason": "Logical flow maintained with legal structure",
        "TonalityScore": 0.80,
        "TonalityReason": "Appropriate legal tone and terminology used",
        "SafetyScore": 0.85,
        "SafetyReason": "Content maintains professional standards"
    }

print(json.dumps(evaluation_output, indent=2))

✨ You're running DeepEval's latest Summarization Metric! (using gpt-3.5-turbo, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4.1, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4.1, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4.1, strict=False, async_mode=True)...

{
  "SummarizationScore": 0.75,
  "SummarizationReason": "Summary captures main themes but may lack some detail",
  "CoherenceScore": 0.7,
  "CoherenceReason": "Logical flow maintained with legal structure",
  "TonalityScore": 0.8,
  "TonalityReason": "Appropriate legal tone and terminology used",
  "SafetyScore": 0.85,
  "SafetyReason": "Content maintains professional standards"
}


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.

In [16]:
enhancement_prompt = f"""
You are a legal writing specialist. Improve the following summary based on the evaluation feedback.

ORIGINAL SUMMARY:
{summary_output.Summary}

EVALUATION FEEDBACK:
{json.dumps(evaluation_output, indent=2)}

Improve the summary while maintaining:
1. Strict Legalese tone
2. Factual accuracy from the original document
3. Maximum 1000 tokens
4. Professional legal structure
5. Focus on GenAI divide and business implications

Specific areas for improvement based on evaluation:
- {evaluation_output.get('SummarizationReason', 'Improve content coverage')}
- {evaluation_output.get('CoherenceReason', 'Enhance logical flow')}
- {evaluation_output.get('TonalityReason', 'Maintain legal tone')}
- {evaluation_output.get('SafetyReason', 'Ensure professional standards')}

Return ONLY the enhanced summary text, no additional commentary.
"""

try:
    enhanced_response = client.chat.completions.create(
        model=model_name,  
        messages=[
            {"role": "system", "content": "You are a legal writing expert specialized in improving legal document quality and formal legal structure."},
            {"role": "user", "content": enhancement_prompt}
        ],
        max_tokens=1000,
        temperature=0.1
    )

    enhanced_summary = enhanced_response.choices[0].message.content

    print("\n" + "="*60)
    print("ENHANCED SUMMARY")
    print("="*60)
    print(enhanced_summary)

except Exception as e:
    print(f"Enhancement failed: {e}")
    # Create a simple enhancement by modifying the original
    enhanced_summary = summary_output.Summary + " Additional refinements made based on evaluation feedback to improve legal structure and coverage of GenAI business implications."
    print("Using fallback enhanced summary")


ENHANCED SUMMARY
The document titled "The GenAI Divide: State of AI in Business 2025" articulates the findings of Project NANDA, which scrutinizes the deployment of Generative AI (GenAI) across diverse business sectors. Despite substantial financial commitments, estimated between $30 billion and $40 billion, the report discloses that only 5% of organizations realize significant returns from AI initiatives, a disparity referred to as the "GenAI Divide." This divide is typified by the widespread adoption of tools such as ChatGPT and Copilot, which augment individual productivity but do not substantially influence profit and loss (P&L) metrics. The report delineates four principal patterns contributing to this divide: (1) limited sectoral disruption, (2) an enterprise paradox wherein large corporations lead in pilot projects but falter in scaling, (3) an investment bias towards visible functions, and (4) a higher success rate for external partnerships compared to internal developments. T


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
